# 03 — Prepare CB2 molecular modeling inputs

This notebook verifies one curated human CB2 affinity dataset, builds fixed molecular fingerprints and scaffold labels, audits structure identity, and freezes random and scaffold splits. It then audits the saved boundaries and writes a completed preparation handoff. Model fitting and predictions belong to the following model notebooks.

**Reference:** All source and supporting files are in [provenance](../provenance/README.md). The data provenance [Excel workbook](../provenance/CB2_data_provenance.xlsx) is the supplied presentation snapshot; reruns preserve it unchanged. Saved outputs below describe the previous execution; printed old paths can differ. Rerun notebooks 00–09 in order to refresh the complete pipeline.


## 1. Select and verify one completed curation run

The manifest names the exact input files and records their SHA-256 hashes. A hash compares current file bytes with the completed curation record, so a changed or missing source stops this run. The working directory may be the repository root or its `notebooks` folder.

In [1]:
# Import file/provenance tools, molecular representations, and split utilities.
from pathlib import Path
import sys
# Shared storage helpers keep the notebooks on the same completed dataset.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "scripts"))
from provenance_support import (
    latest_acquisition, current_selection, current_curation, current_preparation,
    current_mlp_baseline, save_review_record, load_review_record,
)
from datetime import datetime, timezone
from itertools import combinations
import hashlib
import json
import platform
import subprocess

import numpy as np
import pandas as pd
import rdkit
import sklearn
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit, train_test_split

CURATION_MANIFEST_REL = current_curation(project_root)
# Set this to an explicit repository-relative preparation folder to resume an interrupted run.
# Leave it as None to create a separate numbered experiment; existing splits are never replaced.
EXISTING_PREP_RUN_REL = None
run_started_at_utc = datetime.now(timezone.utc).isoformat()

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "notebooks/02_data_curation.ipynb").is_file(), (
    "Run from the repository root or its notebooks folder."
)

def sha256_file(path):
    """Hash a file in chunks so large source files do not fill memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Require a completed, audited human CB2 handoff before trusting its paths or modeling tables.
curation_manifest_path = project_root / CURATION_MANIFEST_REL
assert curation_manifest_path.is_file(), f"Required curation manifest is missing: {curation_manifest_path}"
with curation_manifest_path.open(encoding="utf-8") as stream:
    curation_manifest = json.load(stream)
assert curation_manifest.get("completed_at_utc"), "Curation manifest has no completion timestamp."
required_upstream_checks = (
    "source_values", "pki", "structure", "quality_flags", "aggregation",
    "workbook_summary", "saved_files",
)
upstream_results = curation_manifest.get("audit", {}).get("results", {})
for check in required_upstream_checks:
    assert upstream_results.get(check) is True, f"Upstream curation check is absent or false: {check}"
assert (
    curation_manifest.get("target_chembl_id"),
    curation_manifest.get("target_organism"),
    curation_manifest.get("uniprot"),
) == ("CHEMBL253", "Homo sapiens", "P34972"), "Unexpected target identity in curation."

# Check that the manifest, selected data, and raw snapshot all refer to the same acquisition.
output_folder_rel = Path(curation_manifest["output_folder"])
assert (project_root / output_folder_rel).resolve() == curation_manifest_path.parent.resolve(), (
    "Curation output_folder differs from the manifest location."
)
# Check acquisition identity through source metadata; output folders now use descriptive names.
with (project_root / curation_manifest["selection_manifest"]).open(encoding="utf-8") as stream:
    selection_source = json.load(stream)
assert Path(selection_source["source_raw_folder"]) == Path(curation_manifest["source_raw_folder"])
assert Path(selection_source["source_manifest"]) == Path(curation_manifest["acquisition_manifest"])
assert Path(curation_manifest["source_raw_folder"]).name in Path(curation_manifest["selection_manifest"]).parts
assert Path(curation_manifest["acquisition_manifest"]).parent == Path(curation_manifest["source_raw_folder"])
assert Path(curation_manifest["selected_csv"]).parent == Path(curation_manifest["selection_manifest"]).parent

# Each declared source and output is checked, not just the two files used for modeling.
source_hashes = curation_manifest["file_sha256"]
for relative_name, recorded_hash in source_hashes.items():
    source_path = project_root / relative_name
    assert source_path.is_file(), f"Curation source is missing: {relative_name}"
    actual_hash = sha256_file(source_path)
    assert actual_hash == recorded_hash, f"Curation source hash mismatch: {relative_name}"
for recorded_name in ("selection_manifest", "selected_csv", "acquisition_manifest"):
    assert curation_manifest[recorded_name] in source_hashes, f"Unhashed curation source: {recorded_name}"
for filename in curation_manifest["output_schemas"]:
    assert (output_folder_rel / filename).as_posix() in source_hashes, f"Unhashed curation output: {filename}"

curated_name = "curated_structures.csv"
bridge_name = "measurement_decisions.csv"
assert curated_name in curation_manifest["output_schemas"]
assert bridge_name in curation_manifest["output_schemas"]
curated_path = project_root / output_folder_rel / curated_name
bridge_path = project_root / output_folder_rel / bridge_name
print(f"{'Curation manifest':<25} | {CURATION_MANIFEST_REL}")
print(f"{'Verified source files':<25} | {len(source_hashes):>8,}")
print(f"{'Curated CSV SHA-256':<25} | {source_hashes[(output_folder_rel / curated_name).as_posix()]}")
print(f"{'Bridge CSV SHA-256':<25} | {source_hashes[(output_folder_rel / bridge_name).as_posix()]}")

Curation manifest         | provenance/curation/run_01/manifest.json
Verified source files     |        9
Curated CSV SHA-256       | 80f3adcb6598c18b67c51612a798dd0199231851fd68a9af20e4e2d5bcdab953
Bridge CSV SHA-256        | f6e726424fff791cb72801e4511aaf41ecdf2ad3aa0511dfad6a629dcb820a7a


The curated CSV supplies one target per full canonical isomeric SMILES, while the bridge retains links to activity, assay, document, and molecule IDs. This cell checks column schemas, unique keys and activity IDs, finite values, counts, and the target aggregation independently. Byte integrity and internal agreement do not establish that the original experiments were scientifically correct.

In [2]:
# Read fields as text first so missing values, identifiers, and numeric conversions are controlled.
curated_columns = [
    "rdkit_smiles", "pki_target", "n_measurements", "n_assays", "n_documents",
    "n_molecule_ids", "pki_min", "pki_max", "pki_range", "pki_overall_median",
]
assert curation_manifest["output_schemas"][curated_name] == curated_columns
curated = pd.read_csv(curated_path, dtype=str, keep_default_na=False)
bridge = pd.read_csv(bridge_path, dtype=str, keep_default_na=False)
assert list(curated.columns) == curated_columns, "Curated CSV columns differ from the manifest."
assert list(bridge.columns) == curation_manifest["output_schemas"][bridge_name], (
    "Measurement bridge columns differ from the manifest."
)
assert curated["rdkit_smiles"].str.strip().ne("").all()
assert curated["rdkit_smiles"].is_unique, "Repeated canonical-isomeric key in curated CSV."
assert bridge["activity_id"].str.strip().ne("").all()
assert bridge["activity_id"].is_unique, "Activity ID occurs more than once in the bridge."
assert set(bridge["status"]) == set(curation_manifest["final_status_counts"])

# Validate evidence counts and finite target summaries; a malformed value stops the handoff.
integer_columns = ["n_measurements", "n_assays", "n_documents", "n_molecule_ids"]
numeric_columns = ["pki_target", "pki_min", "pki_max", "pki_range", "pki_overall_median"]
for column in integer_columns:
    assert curated[column].str.fullmatch(r"[0-9]+").all(), f"Invalid integer count: {column}"
    curated[column] = pd.to_numeric(curated[column], errors="raise").astype("int64")
for column in numeric_columns:
    curated[column] = pd.to_numeric(curated[column], errors="raise")
    assert np.isfinite(curated[column]).all(), f"Nonfinite curated value: {column}"
assert curated["n_measurements"].ge(1).all()
assert curated["n_assays"].ge(1).all()
assert curated["n_molecule_ids"].ge(1).all()
assert curated["n_documents"].ge(0).all()
for column in ("n_assays", "n_documents", "n_molecule_ids"):
    assert curated[column].le(curated["n_measurements"]).all(), column
assert curated["pki_max"].ge(curated["pki_min"]).all()
np.testing.assert_allclose(
    curated["pki_range"], curated["pki_max"] - curated["pki_min"], rtol=0, atol=1e-10
)
assert curated["pki_target"].between(curated["pki_min"] - 1e-10, curated["pki_max"] + 1e-10).all()

# Use only retained measurements, then reconcile their structure keys and totals with curation.
retained = bridge.loc[bridge["status"].eq("retained")].copy()
assert retained["rdkit_smiles"].str.strip().ne("").all()
assert retained["assay_chembl_id"].str.strip().ne("").all()
assert retained["molecule_chembl_id"].str.strip().ne("").all()
assert set(retained["rdkit_smiles"]) == set(curated["rdkit_smiles"]), (
    "Retained bridge keys differ from curated structure keys."
)
retained["pki"] = pd.to_numeric(retained["pki"], errors="raise")
assert np.isfinite(retained["pki"]).all()
counts = curation_manifest["counts"]
assert len(bridge) == counts["input_activities"]
assert len(retained) == counts["retained_activities"]
assert len(curated) == counts["final_curated_structures"]
assert int(curated["n_measurements"].sum()) == len(retained)
assert int(curated["n_measurements"].eq(1).sum()) == counts["single_measurement_structures"]
for status, expected_count in curation_manifest["final_status_counts"].items():
    assert int(bridge["status"].eq(status).sum()) == expected_count, status

# Recompute each target from retained activity pKi: first per assay, then across assays.
assay_medians = retained.groupby(["rdkit_smiles", "assay_chembl_id"])["pki"].median()
rebuilt_targets = assay_medians.groupby(level="rdkit_smiles").median()
curated_by_key = curated.set_index("rdkit_smiles")
assert set(rebuilt_targets.index) == set(curated_by_key.index)
for key, expected in rebuilt_targets.items():
    assert np.isclose(curated_by_key.at[key, "pki_target"], expected, rtol=0, atol=1e-10), key
for key, rows in retained.groupby("rdkit_smiles"):
    saved = curated_by_key.loc[key]
    assert saved["n_measurements"] == len(rows), key
    assert saved["n_assays"] == rows["assay_chembl_id"].nunique(), key
    assert saved["n_molecule_ids"] == rows["molecule_chembl_id"].nunique(), key
    assert saved["n_documents"] == rows.loc[rows["document_chembl_id"].ne(""), "document_chembl_id"].nunique(), key

# Sorting the table once gives every later array and mask one stable row order.
curated = curated.sort_values("rdkit_smiles").reset_index(drop=True)
structure_keys = curated["rdkit_smiles"].to_numpy(dtype=str)
y = curated["pki_target"].to_numpy(dtype=np.float64)
assert np.isfinite(y).all()
print(f"{'Target':<25} | median of within-assay median pKi (Ki in nM: 9 - log10(Ki))")
print(f"{'Modeling structures':<25} | {len(curated):>8,}")
print(f"{'Retained measurements':<25} | {len(retained):>8,}")
print("Handoff checks passed: schema, hashes, identities, counts, evidence, and targets.")

Target                    | median of within-assay median pKi (Ki in nM: 9 - log10(Ki))
Modeling structures       |    3,576
Retained measurements     |    3,905
Handoff checks passed: schema, hashes, identities, counts, evidence, and targets.


## 2. Fix representation and split settings

Morgan bits and Murcko scaffold extraction are deterministic transformations of each molecule, so they can be computed before assigning subsets. The random protocol splits molecular rows; the scaffold protocol splits whole chirality-ignored scaffold groups. The similarity and target-gap thresholds are fixed diagnostics for this preparation run, not universal chemical definitions.

In [3]:
# Describe each structure with radius-2 Morgan neighborhoods in a 2,048-bit binary vector.
FP_RADIUS = 2
FP_BITS = 2048
FP_INCLUDE_CHIRALITY = True
# Keep all ring-free molecules in one scaffold group rather than splitting them individually.
ACYCLIC_LABEL = "ACYCLIC"
# Fix the two random choices: reserve 20%, then divide that reserve into validation and test.
OUTER_SEED = 42
INNER_SEED = 43
OUTER_HELDOUT_FRACTION = 0.20
INNER_TEST_FRACTION = 0.50
# Set diagnostic cutoffs before training; these flags do not remove structures or tune the model.
HIGH_SIMILARITY = 0.80
IDENTICAL_FP_TARGET_GAP = 1.0
SUBSETS = ("train", "validation", "test")
STRATEGIES = ("random", "scaffold")
print("Split sequence: outer seed 42 reserves 20%; inner seed 43 halves that reserve.")
print("No estimator settings or model fitting enter this preparation stage.")

Split sequence: outer seed 42 reserves 20%; inner seed 43 halves that reserve.
No estimator settings or model fitting enter this preparation stage.


Each curated key is reparsed as one sanitized, connected molecule and round-tripped to the same canonical isomeric SMILES. A mismatch stops the run because dropping or rewriting a structure here would change the curated dataset. Only 2,048 binary fingerprint bits become predictors; evidence counts, labels, and target summaries remain metadata.

In [4]:
# Create one deterministic fingerprint generator with the declared radius, length, and chirality.
fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS, fpSize=FP_BITS, includeChirality=FP_INCLUDE_CHIRALITY
)
molecules = []
fingerprints = []
scaffolds = []
# Allocate one binary feature row per sorted structure key; this order also aligns with y.
X = np.zeros((len(structure_keys), FP_BITS), dtype=np.uint8)
for row_number, key in enumerate(structure_keys):
    # Recheck each curated structure without changing its chemistry or silently dropping failures.
    molecule = Chem.MolFromSmiles(key, sanitize=True)
    assert molecule is not None and molecule.GetNumAtoms() > 0, f"Invalid curated SMILES: {key}"
    assert len(Chem.GetMolFrags(molecule)) == 1, f"Multiple components: {key}"
    assert all(atom.GetAtomicNum() > 0 for atom in molecule.GetAtoms()), f"Wildcard atom: {key}"
    round_trip = Chem.MolToSmiles(molecule, canonical=True, isomericSmiles=True)
    assert round_trip == key, f"RDKit canonical round-trip changed: {key} -> {round_trip}"
    # Store the same fingerprint as an RDKit bit vector for similarity and a NumPy row for training.
    fingerprint = fingerprint_generator.GetFingerprint(molecule)
    DataStructs.ConvertToNumpyArray(fingerprint, X[row_number])
    # Extract a chirality-ignored ring/linker framework; an empty framework becomes ACYCLIC.
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=molecule, includeChirality=False)
    molecules.append(molecule)
    fingerprints.append(fingerprint)
    scaffolds.append(scaffold or ACYCLIC_LABEL)

# Check binary values and row alignment before the representations are used in any audit or split.
assert X.shape == (len(curated), FP_BITS)
assert np.isin(X, (0, 1)).all()
assert len(fingerprints) == len(scaffolds) == len(y) == len(structure_keys)
assert np.array_equal(structure_keys, curated["rdkit_smiles"].to_numpy())
print(f"{'RDKit version':<26} | {rdkit.__version__}")
print(f"{'Fingerprint matrix':<26} | {X.shape[0]:,} structures × {X.shape[1]:,} bits")
print(f"{'Scaffold groups':<26} | {len(set(scaffolds)):>8,}")
print(f"{'Largest scaffold group':<26} | {pd.Series(scaffolds).value_counts().iloc[0]:>8,} structures")
print(f"{'Acyclic structures':<26} | {scaffolds.count(ACYCLIC_LABEL):>8,}")

RDKit version              | 2026.03.6
Fingerprint matrix         | 3,576 structures × 2,048 bits
Scaffold groups            |    1,358
Largest scaffold group     |       93 structures
Acyclic structures         |       29


## 3. Audit A — identity and representation before splitting

One canonical key must correspond to one modeling row, and each retained activity must trace to that row. Molecule IDs can point to more than one represented key, and keys can have evidence from multiple molecule IDs; these are reported for review without merging them. Preserved protonation, tautomers, and unspecified stereochemistry can also leave related representations separate.

In [5]:
# Confirm one finite target per structure and one structure assignment per retained activity.
assert len(structure_keys) == len(set(structure_keys))
assert np.isfinite(y).all()
activity_key_counts = retained.groupby("activity_id")["rdkit_smiles"].nunique()
assert activity_key_counts.eq(1).all(), "A retained activity maps to multiple structures."
# Inspect database-ID-to-structure relationships without assuming that an ID is the structure key.
molecule_to_keys = retained.groupby("molecule_chembl_id")["rdkit_smiles"].nunique()
keys_to_molecules = retained.groupby("rdkit_smiles")["molecule_chembl_id"].nunique()
multi_key_molecule_ids = molecule_to_keys.loc[molecule_to_keys.gt(1)]
multi_id_keys = keys_to_molecules.loc[keys_to_molecules.gt(1)]
# Match the complete quality-flag token and count affected measurements separately from structures.
unspecified_rows = retained["quality_flags"].str.contains(
    r"(?:^|;)potential_unspecified_stereo(?:;|$)", regex=True
)
unspecified_keys = set(retained.loc[unspecified_rows, "rdkit_smiles"])
# Reconcile single-measurement structures; one observation cannot establish experimental agreement.
assert int(curated["n_measurements"].eq(1).sum()) == counts["single_measurement_structures"]
# Display evidence limitations and any ID ambiguities without excluding otherwise retained rows.
print(f"{'Single-measurement structures':<38} | {curated['n_measurements'].eq(1).sum():>8,}")
print(f"{'Unspecified-stereo activity rows':<38} | {unspecified_rows.sum():>8,}")
print(f"{'Unspecified-stereo structures':<38} | {len(unspecified_keys):>8,}")
print(f"{'Molecule IDs spanning multiple keys':<38} | {len(multi_key_molecule_ids):>8,}")
print(f"{'Keys with multiple molecule IDs':<38} | {len(multi_id_keys):>8,}")
if len(multi_key_molecule_ids):
    print("Example molecule IDs for source review:", multi_key_molecule_ids.head(5).to_dict())

Single-measurement structures          |    3,263
Unspecified-stereo activity rows       |      871
Unspecified-stereo structures          |      844
Molecule IDs spanning multiple keys    |        0
Keys with multiple molecule IDs        |        0


A Morgan fingerprint compresses structure into a fixed number of bits, so distinct curated structures can have the same input vector. The complete byte sequence defines each group; a SHA-256 digest is only a convenient stable label and is checked against the actual arrays. These structures remain separate, even when their targets differ by at least one pKi unit.

In [6]:
# Use a digest to label groups, then verify real bit equality so the digest is not the proof.
fingerprint_group_ids = [hashlib.sha256(row.tobytes()).hexdigest() for row in X]
indices_by_group = {}
for index, group_id in enumerate(fingerprint_group_ids):
    indices_by_group.setdefault(group_id, []).append(index)
for group_id, members in indices_by_group.items():
    for index in members[1:]:
        assert np.array_equal(X[members[0]], X[index]), f"Fingerprint digest collision: {group_id}"

# Prepare separate tables for group-level findings and the individual structures behind them.
group_summary_rows = []
group_member_rows = []
for group_id, members in sorted(indices_by_group.items()):
    if len(members) < 2:
        continue
    # For each repeated fingerprint, summarize target spread, scaffold agreement, and unordered pairs.
    member_keys = [structure_keys[index] for index in members]
    member_targets = y[members]
    member_scaffolds = {scaffolds[index] for index in members}
    gap = float(member_targets.max() - member_targets.min())
    group_summary_rows.append({
        "fingerprint_group_id": group_id,
        "structure_count": len(members),
        "unique_key_count": len(set(member_keys)),
        "target_min": float(member_targets.min()),
        "target_max": float(member_targets.max()),
        "target_range": gap,
        "same_scaffold": len(member_scaffolds) == 1,
        "target_gap_ge_1": gap >= IDENTICAL_FP_TARGET_GAP,
        "unordered_pair_count": len(members) * (len(members) - 1) // 2,
    })
    # Keep each distinct structure and attach its original activity, assay, document, and molecule IDs.
    for index in members:
        key = structure_keys[index]
        source_rows = retained.loc[retained["rdkit_smiles"].eq(key)]
        group_member_rows.append({
            "fingerprint_group_id": group_id,
            "rdkit_smiles": key,
            "pki_target": float(y[index]),
            "scaffold": scaffolds[index],
            "n_measurements": int(len(source_rows)),
            "activity_ids": ";".join(sorted(source_rows["activity_id"])),
            "assay_ids": ";".join(sorted(set(source_rows["assay_chembl_id"]))),
            "document_ids": ";".join(sorted(set(source_rows["document_chembl_id"]) - {""})),
            "molecule_ids": ";".join(sorted(set(source_rows["molecule_chembl_id"]))),
            "potential_unspecified_stereo": key in unspecified_keys,
        })

# Declare schemas explicitly so even an empty diagnostic report retains meaningful column headers.
group_summary_columns = [
    "fingerprint_group_id", "structure_count", "unique_key_count", "target_min",
    "target_max", "target_range", "same_scaffold", "target_gap_ge_1", "unordered_pair_count",
]
group_member_columns = [
    "fingerprint_group_id", "rdkit_smiles", "pki_target", "scaffold", "n_measurements",
    "activity_ids", "assay_ids", "document_ids", "molecule_ids", "potential_unspecified_stereo",
]
group_summary = pd.DataFrame(group_summary_rows, columns=group_summary_columns)
group_members = pd.DataFrame(group_member_rows, columns=group_member_columns)
assert group_summary["unique_key_count"].eq(group_summary["structure_count"]).all()
assert len(group_members) == int(group_summary["structure_count"].sum())
# Report identity checks separately from representation limits; an identical fingerprint is not a duplicate molecule.
print("AUDIT A SUMMARY")
print("Integrity passed: unique keys, finite targets, source-to-target reconciliation, and equal digest-group bits.")
print(f"{'Identical-fingerprint groups':<38} | {len(group_summary):>8,}")
print(f"{'Participating structures':<38} | {len(group_members):>8,}")
print(f"{'Unique unordered pairs':<38} | {group_summary['unordered_pair_count'].sum():>8,}")
print(f"{'Groups with target range >= 1 pKi':<38} | {group_summary['target_gap_ge_1'].sum():>8,}")
print("Policy: retain distinct keys and targets; fingerprint matches are representation findings.")
print("Limit: canonical SMILES equality does not detect every chemical identity relationship.")

AUDIT A SUMMARY
Integrity passed: unique keys, finite targets, source-to-target reconciliation, and equal digest-group bits.
Identical-fingerprint groups           |      142
Participating structures               |      313
Unique unordered pairs                 |      205
Groups with target range >= 1 pKi      |       20
Policy: retain distinct keys and targets; fingerprint matches are representation findings.
Limit: canonical SMILES equality does not detect every chemical identity relationship.


## 4. Create and freeze both split protocols

The random split first reserves 20% of molecular rows and then halves that reserve. The scaffold split applies the same sequence to groups, so molecular proportions may differ. A saved split checkpoint allows an interrupted preparation run to reuse exactly its recorded assignments.

In [7]:
# Record the representation and split rules that must match when an interrupted experiment resumes.
configuration = {
    "fingerprint": {"radius": FP_RADIUS, "bits": FP_BITS, "include_chirality": FP_INCLUDE_CHIRALITY},
    "scaffold": {"method": "RDKit Bemis-Murcko", "include_chirality": False, "acyclic_label": ACYCLIC_LABEL},
    "split": {
        "strategies": list(STRATEGIES), "outer_seed": OUTER_SEED, "inner_seed": INNER_SEED,
        "outer_heldout_fraction": OUTER_HELDOUT_FRACTION,
        "inner_test_fraction": INNER_TEST_FRACTION,
        "random_unit": "molecular row", "scaffold_unit": "scaffold group",
    },
    "diagnostics": {"high_tanimoto_at_least": HIGH_SIMILARITY,
                    "identical_fingerprint_target_gap_at_least_pki": IDENTICAL_FP_TARGET_GAP},
}
modeling_parent = project_root / "provenance/preparation"
# Create a fresh run or open one explicitly named incomplete run; never select a folder implicitly.
if EXISTING_PREP_RUN_REL is None:
    # Numbered runs keep previous results intact; dates are recorded in manifest.json.
    modeling_parent.mkdir(parents=True, exist_ok=True)
    # Start above the highest saved run number after superseded runs are removed.
    existing_run_numbers = [
        int(path.name[4:]) for path in modeling_parent.glob("run_*")
        if path.is_dir() and path.name[4:].isdigit()
    ]
    run_number = max(existing_run_numbers, default=0) + 1
    while True:
        run_name = f"run_{run_number:02d}"
        modeling_folder = modeling_parent / run_name
        try:
            modeling_folder.mkdir(exist_ok=False)
            break
        except FileExistsError:
            run_number += 1
    resuming = False
else:
    modeling_folder = (project_root / EXISTING_PREP_RUN_REL).resolve()
    assert modeling_folder.is_dir(), f"Explicit preparation run is missing: {modeling_folder}"
    assert modeling_folder.parent.resolve() == modeling_parent.resolve()
    assert not (modeling_folder / "manifest.json").exists(), "Run is already complete; create a new experiment."
    resuming = True

# Keep fingerprint-equivalence reports together, keyed to this preparation run.
# The frozen feature arrays and split assignments remain in processed data.
fingerprint_report_folder = project_root / "provenance/preparation" / modeling_folder.name / "identical_fingerprints"
fingerprint_report_folder.mkdir(parents=True, exist_ok=EXISTING_PREP_RUN_REL is not None)

features_path = modeling_folder / "fingerprints_and_targets.npz"
group_summary_path = fingerprint_report_folder / "fingerprint_groups.csv"
group_members_path = fingerprint_report_folder / "fingerprint_group_members.csv"
assignment_path = modeling_folder / "split_assignments.csv"
split_manifest_path = modeling_folder / "split_manifest.json"
curation_manifest_hash = sha256_file(curation_manifest_path)

# A resumed checkpoint must match its source hashes, configuration, and frozen assignment bytes.
if resuming:
    assert all(path.is_file() for path in (
        features_path, group_summary_path, group_members_path, assignment_path, split_manifest_path
    )), "Incomplete split checkpoint: required files are missing."
    with split_manifest_path.open(encoding="utf-8") as stream:
        split_manifest = json.load(stream)
    assert split_manifest["curation_manifest_sha256"] == curation_manifest_hash
    assert split_manifest["source_sha256"] == source_hashes
    assert split_manifest["configuration"] == configuration, "Split settings changed; create a new run."
    assert split_manifest["assignment_sha256"] == sha256_file(assignment_path), (
        "Saved assignments differ from their frozen checkpoint."
    )
else:
    np.savez_compressed(features_path, X=X, y=y, rdkit_smiles=structure_keys)
    group_summary.to_csv(group_summary_path, index=False)
    group_members.to_csv(group_members_path, index=False)

    # Split molecular-row indices twice for the random protocol, keeping X, y, and keys aligned.
    all_indices = np.arange(len(structure_keys))
    random_train, random_reserved = train_test_split(
        all_indices, test_size=OUTER_HELDOUT_FRACTION, random_state=OUTER_SEED, shuffle=True
    )
    random_validation, random_test = train_test_split(
        random_reserved, test_size=INNER_TEST_FRACTION, random_state=INNER_SEED, shuffle=True
    )
    # Split whole scaffold groups, then map the second split from reserved-subset positions to original rows.
    group_labels = np.asarray(scaffolds, dtype=str)
    assert len(set(group_labels)) >= 3, "Too few scaffold groups for three subsets."
    outer_groups = GroupShuffleSplit(n_splits=1, test_size=OUTER_HELDOUT_FRACTION, random_state=OUTER_SEED)
    scaffold_train, scaffold_reserved = next(outer_groups.split(X, y, groups=group_labels))
    inner_groups = GroupShuffleSplit(n_splits=1, test_size=INNER_TEST_FRACTION, random_state=INNER_SEED)
    inner_validation, inner_test = next(inner_groups.split(
        X[scaffold_reserved], y[scaffold_reserved], groups=group_labels[scaffold_reserved]
    ))
    scaffold_validation = scaffold_reserved[inner_validation]
    scaffold_test = scaffold_reserved[inner_test]
    chosen_indices = {
        "random": {"train": random_train, "validation": random_validation, "test": random_test},
        "scaffold": {"train": scaffold_train, "validation": scaffold_validation, "test": scaffold_test},
    }
    # Save complete membership and actual sizes for both protocols before any boundary audit or fitting.
    assignment_rows = []
    actual_proportions = {}
    for strategy in STRATEGIES:
        labels = np.full(len(structure_keys), "", dtype="<U10")
        actual_proportions[strategy] = {}
        for subset in SUBSETS:
            indices = chosen_indices[strategy][subset]
            assert len(indices) > 0, f"Empty {strategy} {subset} subset."
            assert np.all(labels[indices] == ""), f"Duplicate {strategy} assignment."
            labels[indices] = subset
            actual_proportions[strategy][subset] = {
                "structures": int(len(indices)), "fraction": float(len(indices) / len(labels)),
                "scaffold_groups": int(len(set(group_labels[indices]))),
            }
        assert np.all(labels != ""), f"Missing {strategy} assignments."
        for index, key in enumerate(structure_keys):
            assignment_rows.append({
                "split_strategy": strategy, "rdkit_smiles": key, "subset": labels[index],
                "scaffold": scaffolds[index], "fingerprint_group_id": fingerprint_group_ids[index],
            })
    assignments = pd.DataFrame(assignment_rows)
    assignments.to_csv(assignment_path, index=False)
    split_manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_curation_manifest": CURATION_MANIFEST_REL.as_posix(),
        "curation_manifest_sha256": curation_manifest_hash,
        "source_sha256": source_hashes,
        "configuration": configuration,
        "intended_structure_fractions": {"train": 0.8, "validation": 0.1, "test": 0.1},
        "actual_proportions": actual_proportions,
        "assignment_file": assignment_path.relative_to(project_root).as_posix(),
        "assignment_sha256": sha256_file(assignment_path),
        "status": "split_checkpoint_only",
    }
    with split_manifest_path.open("w", encoding="utf-8") as stream:
        json.dump(split_manifest, stream, indent=2)

print(f"{'Preparation run':<24} | {modeling_folder.relative_to(project_root)}")
print(f"{'Checkpoint status':<24} | {'resumed' if resuming else 'created'}")
for strategy, portions in split_manifest["actual_proportions"].items():
    for subset, details in portions.items():
        print(f"{strategy + ' ' + subset:<24} | {details['structures']:>6,} structures ({details['fraction']:.1%}), "
              f"{details['scaffold_groups']:,} groups")


Preparation run          | provenance/preparation/run_01
Checkpoint status        | created
random train             |  2,860 structures (80.0%), 1,186 groups
random validation        |    358 structures (10.0%), 245 groups
random test              |    358 structures (10.0%), 252 groups
scaffold train           |  2,778 structures (77.7%), 1,086 groups
scaffold validation      |    461 structures (12.9%), 136 groups
scaffold test            |    337 structures (9.4%), 136 groups


## 5. Audit B — inspect the saved boundaries before fitting

Assignments are reloaded from disk, so the audit tests the frozen checkpoint rather than an in-memory draft. Every structure must appear once in each protocol, and scaffold groups must remain intact in the scaffold protocol. The activity bridge also confirms that a retained activity cannot cross a boundary.

In [8]:
# Audit assignments reloaded from disk, including their schema, uniqueness, and legal subset labels.
assignments = pd.read_csv(assignment_path, dtype=str, keep_default_na=False)
assignment_columns = ["split_strategy", "rdkit_smiles", "subset", "scaffold", "fingerprint_group_id"]
assert list(assignments.columns) == assignment_columns
assert len(assignments) == len(structure_keys) * len(STRATEGIES)
assert not assignments.duplicated(["split_strategy", "rdkit_smiles"]).any(), (
    "Duplicate strategy/structure assignment."
)
assert set(assignments["split_strategy"]) == set(STRATEGIES)
assert set(assignments["subset"]) == set(SUBSETS)
# Build expected structural metadata from the verified arrays rather than trusting the saved labels.
expected_metadata = pd.DataFrame({
    "rdkit_smiles": structure_keys, "scaffold": scaffolds,
    "fingerprint_group_id": fingerprint_group_ids,
})
split_indices = {}
key_to_index = {key: index for index, key in enumerate(structure_keys)}
subset_for_key = {}
# Check each protocol covers the full dataset and joins one-to-one with the expected structure metadata.
for strategy in STRATEGIES:
    rows = assignments.loc[assignments["split_strategy"].eq(strategy)].copy()
    assert set(rows["rdkit_smiles"]) == set(structure_keys), f"Incomplete {strategy} coverage."
    aligned = expected_metadata.merge(rows, on="rdkit_smiles", suffixes=("_expected", ""), validate="one_to_one")
    assert (aligned["scaffold_expected"] == aligned["scaffold"]).all(), f"Scaffold alignment error: {strategy}"
    assert (aligned["fingerprint_group_id_expected"] == aligned["fingerprint_group_id"]).all(), (
        f"Fingerprint alignment error: {strategy}"
    )
    subset_for_key[strategy] = dict(zip(rows["rdkit_smiles"], rows["subset"]))
    split_indices[strategy] = {}
    # Translate subset keys back to the common sorted array order and reconcile recorded subset sizes.
    for subset in SUBSETS:
        keys = set(rows.loc[rows["subset"].eq(subset), "rdkit_smiles"])
        assert keys, f"Empty {strategy} {subset} subset."
        indices = np.asarray([index for index, key in enumerate(structure_keys) if key in keys], dtype=int)
        assert set(structure_keys[indices]) == keys, f"Index alignment error: {strategy} {subset}"
        split_indices[strategy][subset] = indices
        recorded = split_manifest["actual_proportions"][strategy][subset]
        assert recorded["structures"] == len(indices), f"Checkpoint count mismatch: {strategy} {subset}"
        assert recorded["scaffold_groups"] == len({scaffolds[index] for index in indices})
    assert len(set().union(*(set(structure_keys[split_indices[strategy][subset]]) for subset in SUBSETS))) == len(structure_keys)
    # Trace retained activities through their structure keys to confirm that no activity crosses a boundary.
    activity_subsets = retained["rdkit_smiles"].map(subset_for_key[strategy])
    assert activity_subsets.notna().all(), f"Retained activity lacks a {strategy} assignment."
    assert activity_subsets.groupby(retained["activity_id"]).nunique().eq(1).all(), (
        f"Retained activity crosses {strategy} boundaries."
    )
    # Require scaffold groups to stay intact only for the protocol that promises scaffold separation.
    if strategy == "scaffold":
        assert rows.groupby("scaffold")["subset"].nunique().eq(1).all(), (
            "Scaffold group crosses scaffold-split boundaries."
        )
print("Reloaded assignments pass complete coverage, row alignment, and scaffold-group checks.")

Reloaded assignments pass complete coverage, row alignment, and scaffold-group checks.


Distinct structures with identical fingerprints can land across subsets, even when a scaffold is held together. This cell enumerates exact cross-boundary fingerprint pairs and compares study context through the retained bridge. Shared assays and documents are reported as dependence findings; they do not prove duplicated experiments or trigger resplitting.

In [9]:
# Store structure identifiers for matching pairs, without exporting held-out target differences.
pair_columns = [
    "split_strategy", "left_subset", "right_subset", "left_rdkit_smiles",
    "right_rdkit_smiles", "fingerprint_group_id",
]
pair_rows = []
boundary_summaries = []
# Inspect all three subset boundaries within each protocol and enforce structure disjointness.
for strategy in STRATEGIES:
    for left_subset, right_subset in combinations(SUBSETS, 2):
        left_indices = split_indices[strategy][left_subset]
        right_indices = split_indices[strategy][right_subset]
        left_keys = set(structure_keys[left_indices])
        right_keys = set(structure_keys[right_indices])
        assert left_keys.isdisjoint(right_keys), f"Structure crosses {strategy} boundaries."
        left_scaffolds = {scaffolds[index] for index in left_indices}
        right_scaffolds = {scaffolds[index] for index in right_indices}
        shared_scaffolds = left_scaffolds & right_scaffolds
        if strategy == "scaffold":
            assert not shared_scaffolds, "Scaffold overlap in scaffold protocol."
        # Use the activity bridge to identify shared assay and document context, excluding blank document IDs.
        left_sources = retained.loc[retained["rdkit_smiles"].isin(left_keys)]
        right_sources = retained.loc[retained["rdkit_smiles"].isin(right_keys)]
        shared_assays = set(left_sources["assay_chembl_id"]) & set(right_sources["assay_chembl_id"])
        shared_documents = (set(left_sources["document_chembl_id"]) - {""}) & (
            set(right_sources["document_chembl_id"]) - {""}
        )
        # Count matched pairs and affected molecules separately because one molecule can match several neighbors.
        affected_left = set()
        affected_right = set()
        pair_start = len(pair_rows)
        for group_id, members in indices_by_group.items():
            left_members = [index for index in members if structure_keys[index] in left_keys]
            right_members = [index for index in members if structure_keys[index] in right_keys]
            for left_index in left_members:
                for right_index in right_members:
                    # The group ID was already checked against complete binary-array equality.
                    assert np.array_equal(X[left_index], X[right_index])
                    left_key, right_key = structure_keys[left_index], structure_keys[right_index]
                    pair_rows.append({
                        "split_strategy": strategy, "left_subset": left_subset,
                        "right_subset": right_subset, "left_rdkit_smiles": left_key,
                        "right_rdkit_smiles": right_key, "fingerprint_group_id": group_id,
                    })
                    affected_left.add(left_key)
                    affected_right.add(right_key)
        # Record overlap as context; only a violated split rule triggers failure rather than automatic resplitting.
        boundary_summaries.append({
            "split_strategy": strategy, "left_subset": left_subset, "right_subset": right_subset,
            "identical_fingerprint_pairs": len(pair_rows) - pair_start,
            "affected_left_structures": len(affected_left),
            "affected_right_structures": len(affected_right),
            "shared_scaffold_count": len(shared_scaffolds),
            "shared_assay_count": len(shared_assays),
            "shared_document_count": len(shared_documents),
            "shared_assay_ids": sorted(shared_assays),
            "shared_document_ids": sorted(shared_documents),
        })
cross_pairs = pd.DataFrame(pair_rows, columns=pair_columns).sort_values(pair_columns).reset_index(drop=True)
assert not cross_pairs.duplicated(pair_columns[:-1]).any()
for row in boundary_summaries:
    print(f"{row['split_strategy'] + ' ' + row['left_subset'] + '/' + row['right_subset']:<31} | "
          f"{row['identical_fingerprint_pairs']:>5,} fingerprint pairs | "
          f"{row['shared_scaffold_count']:>5,} shared scaffolds | "
          f"{row['shared_assay_count']:>5,} assays | {row['shared_document_count']:>5,} documents")

random train/validation         |    34 fingerprint pairs |   164 shared scaffolds |   149 assays |   148 documents
random train/test               |    40 fingerprint pairs |   157 shared scaffolds |   163 assays |   161 documents
random validation/test          |     5 fingerprint pairs |    71 shared scaffolds |   111 assays |   115 documents
scaffold train/validation       |    18 fingerprint pairs |     0 shared scaffolds |    99 assays |   101 documents
scaffold train/test             |    12 fingerprint pairs |     0 shared scaffolds |    98 assays |    98 documents
scaffold validation/test        |    10 fingerprint pairs |     0 shared scaffolds |    60 assays |    60 documents


For each validation and test structure, the next cell compares its fingerprint only with that protocol’s training fingerprints. RDKit bulk similarity avoids a full dataset-by-dataset matrix. If several training keys tie for the maximum, the lexicographically first is saved along with the tie count; all exact cross-boundary pairs remain in the separate pair table.

In [10]:
# Define one neighbor record per held-out structure, keeping test targets out of this structural audit.
neighbor_columns = [
    "split_strategy", "subset", "rdkit_smiles", "nearest_training_rdkit_smiles",
    "maximum_tanimoto", "exact_training_fingerprint_match", "nearest_tie_count",
]
neighbor_rows = []
neighbor_summaries = []
# Compare held-out fingerprints only with training fingerprints from the same split protocol.
for strategy in STRATEGIES:
    training_indices = split_indices[strategy]["train"]
    training_keys = structure_keys[training_indices]
    training_fingerprints = [fingerprints[index] for index in training_indices]
    assert list(training_keys) == sorted(training_keys), "Training keys must be lexicographically ordered."
    for subset in ("validation", "test"):
        start = len(neighbor_rows)
        # Calculate one vector of Tanimoto similarities at a time instead of a full all-pairs matrix.
        for index in split_indices[strategy][subset]:
            similarities = np.asarray(
                DataStructs.BulkTanimotoSimilarity(fingerprints[index], training_fingerprints),
                dtype=np.float64,
            )
            maximum = float(similarities.max())
            # Resolve equal maxima by the existing lexical key order and retain how many training neighbors tied.
            tied = np.flatnonzero(similarities == maximum)
            assert len(tied) >= 1
            nearest_key = training_keys[tied[0]]
            neighbor_rows.append({
                "split_strategy": strategy, "subset": subset, "rdkit_smiles": structure_keys[index],
                "nearest_training_rdkit_smiles": nearest_key,
                "maximum_tanimoto": maximum,
                "exact_training_fingerprint_match": bool(maximum == 1.0),
                "nearest_tie_count": int(len(tied)),
            })
        # Summarize nearest-training similarity, exact matches, and the predeclared high-similarity cutoff.
        values = np.asarray([row["maximum_tanimoto"] for row in neighbor_rows[start:]])
        neighbor_summaries.append({
            "split_strategy": strategy, "subset": subset, "structures": int(len(values)),
            "maximum_similarity_min": float(values.min()),
            "maximum_similarity_q25": float(np.quantile(values, 0.25)),
            "maximum_similarity_median": float(np.median(values)),
            "maximum_similarity_q75": float(np.quantile(values, 0.75)),
            "maximum_similarity_max": float(values.max()),
            "exact_training_matches": int(np.count_nonzero(values == 1.0)),
            "high_similarity_structures": int(np.count_nonzero(values >= HIGH_SIMILARITY)),
        })
neighbors = pd.DataFrame(neighbor_rows, columns=neighbor_columns).sort_values(
    ["split_strategy", "subset", "rdkit_smiles"]
).reset_index(drop=True)
for row in neighbor_summaries:
    print(f"{row['split_strategy'] + ' ' + row['subset']:<25} | "
          f"median nearest Tanimoto {row['maximum_similarity_median']:.3f} | "
          f"{row['exact_training_matches']:>4,} exact | "
          f"{row['high_similarity_structures']:>4,} >= {HIGH_SIMILARITY:.2f}")

random validation         | median nearest Tanimoto 0.799 |   27 exact |  179 >= 0.80
random test               | median nearest Tanimoto 0.803 |   31 exact |  185 >= 0.80
scaffold validation       | median nearest Tanimoto 0.672 |   18 exact |   86 >= 0.80
scaffold test             | median nearest Tanimoto 0.712 |   12 exact |   62 >= 0.80


The two audits distinguish blocking integrity failures from reportable structural findings. This cell saves the source-linked findings and held-out structural similarities after reloading and checking the frozen assignments. No fitted model or prediction is produced in this notebook.

In [11]:
# Record checks that have completed in the preceding cells; these flags assume top-to-bottom execution.
audit_a = {
    "blocking_checks": {
        "unique_structure_keys": True, "finite_targets": True,
        "retained_bridge_and_target_reconciled": True, "fingerprint_bits_confirmed": True,
    },
    # Keep observed evidence and fingerprint limitations distinct from the blocking integrity checks.
    "findings": {
        "single_measurement_structures": int(curated["n_measurements"].eq(1).sum()),
        "unspecified_stereo_activity_rows": int(unspecified_rows.sum()),
        "unspecified_stereo_structures": len(unspecified_keys),
        "molecule_ids_with_multiple_keys": len(multi_key_molecule_ids),
        "keys_with_multiple_molecule_ids": len(multi_id_keys),
        "identical_fingerprint_groups": len(group_summary),
        "identical_fingerprint_structures": len(group_members),
        "identical_fingerprint_unordered_pairs": int(group_summary["unordered_pair_count"].sum()),
        "groups_with_target_range_ge_1_pki": int(group_summary["target_gap_ge_1"].sum()),
    },
    "policy": "Distinct curated keys stay separate; fingerprints do not define structure identity.",
    "limitation": "Canonical SMILES does not resolve all tautomer, protonation, or unspecified stereo relationships.",
}
# Preserve boundary findings and explain that scaffold separation does not eliminate all similarity.
audit_b = {
    "blocking_checks": {
        "saved_assignments_reloaded": True, "complete_unique_assignments": True,
        "aligned_structure_keys_and_fingerprints": True,
        "no_activity_crosses_boundaries": True, "scaffold_protocol_group_disjointness": True,
    },
    "boundaries": boundary_summaries,
    "nearest_training_similarity": neighbor_summaries,
    "limitations": [
        "Scaffold disjointness does not imply low fingerprint similarity.",
        "Shared assay and document IDs indicate context, not proven duplicate experiments.",
        "Test reservation is protocol-specific; a structure may train one protocol and be held out in another.",
        "No held-out target gaps or prediction metrics were inspected.",
    ],
}
# Write complete audit and neighbor reports beside the frozen split checkpoint for later evaluation.
audit_summary = {"audit_a": audit_a, "audit_b": audit_b}
cross_pairs_path = fingerprint_report_folder / "cross_split_fingerprint_pairs.csv"
neighbors_path = modeling_folder / "heldout_training_neighbors.csv"
audit_summary_path = modeling_folder / "audit_summary.json"
cross_pairs.to_csv(cross_pairs_path, index=False)
neighbors.to_csv(neighbors_path, index=False)
with audit_summary_path.open("w", encoding="utf-8") as stream:
    json.dump(audit_summary, stream, indent=2)
# Summarize the completed audit without changing assignments in response to reported overlap.
print("AUDIT B SUMMARY")
print("Blocking checks passed: coverage, structure and activity disjointness, alignment, and scaffold grouping.")
print(f"{'Cross-boundary fingerprint pairs':<38} | {len(cross_pairs):>8,}")
print(f"{'Held-out neighbor rows':<38} | {len(neighbors):>8,}")
print("Findings are saved without changing the frozen assignments.")

AUDIT B SUMMARY
Blocking checks passed: coverage, structure and activity disjointness, alignment, and scaffold grouping.
Cross-boundary fingerprint pairs       |      119
Held-out neighbor rows                 |    1,514
Findings are saved without changing the frozen assignments.


## 6. Verify saved preparation artifacts

The saved arrays and reports are reloaded rather than accepted at write time. Every fingerprint bit is regenerated from its saved structure key, and the saved split assignments and audit summaries are reconciled with the audited inputs. These checks make the preparation folder a traceable input for independent model notebooks.

In [12]:
# Reload saved arrays without pickle and compare their types, dimensions, key order, and target values.
with np.load(features_path, allow_pickle=False) as saved:
    saved_X = saved["X"]
    saved_y = saved["y"]
    saved_keys = saved["rdkit_smiles"]
assert saved_X.dtype == np.uint8 and saved_y.dtype == np.float64
assert saved_keys.dtype.kind == "U"
assert saved_X.shape == (len(structure_keys), FP_BITS)
assert saved_y.shape == (len(structure_keys),)
np.testing.assert_array_equal(saved_keys, structure_keys)
np.testing.assert_array_equal(saved_X, X)
np.testing.assert_allclose(saved_y, y, rtol=0, atol=1e-10)
# Regenerate each fingerprint from its saved structure key and compare every bit to the exported row.
for index, key in enumerate(saved_keys):
    molecule = Chem.MolFromSmiles(str(key), sanitize=True)
    assert molecule is not None
    rebuilt = fingerprint_generator.GetFingerprint(molecule)
    bits = np.zeros(FP_BITS, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(rebuilt, bits)
    np.testing.assert_array_equal(bits, saved_X[index], err_msg=f"Saved bits changed for {key}")

# Recheck the frozen assignment hash, full structure coverage, disjoint subsets, and scaffold grouping.
saved_assignments = pd.read_csv(assignment_path, dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(saved_assignments, assignments)
assert split_manifest["assignment_sha256"] == sha256_file(assignment_path)
assert not saved_assignments.duplicated(["split_strategy", "rdkit_smiles"]).any()
for strategy in STRATEGIES:
    rows = saved_assignments.loc[saved_assignments["split_strategy"].eq(strategy)]
    assert set(rows["rdkit_smiles"]) == set(saved_keys)
    assert set(rows["subset"]) == set(SUBSETS)
    for left, right in combinations(SUBSETS, 2):
        assert set(rows.loc[rows["subset"].eq(left), "rdkit_smiles"]).isdisjoint(
            rows.loc[rows["subset"].eq(right), "rdkit_smiles"]
        )
    if strategy == "scaffold":
        assert rows.groupby("scaffold")["subset"].nunique().eq(1).all()

# Reload fingerprint-group reports and check their schemas and expected group/member coverage.
saved_group_summary = pd.read_csv(group_summary_path, dtype=str, keep_default_na=False)
saved_group_members = pd.read_csv(group_members_path, dtype=str, keep_default_na=False)
assert list(saved_group_summary.columns) == group_summary_columns
assert list(saved_group_members.columns) == group_member_columns
assert set(saved_group_summary["fingerprint_group_id"]) == set(group_summary["fingerprint_group_id"])
assert set(zip(saved_group_members["fingerprint_group_id"], saved_group_members["rdkit_smiles"])) == set(
    zip(group_members["fingerprint_group_id"], group_members["rdkit_smiles"])
)
# Confirm actual bit equality, target alignment, and the activity/source IDs behind each saved member.
for group_id, rows in saved_group_members.groupby("fingerprint_group_id"):
    indices = [key_to_index[key] for key in rows["rdkit_smiles"]]
    assert len(indices) >= 2 and len(set(indices)) == len(indices)
    assert all(fingerprint_group_ids[index] == group_id for index in indices)
    assert all(np.array_equal(saved_X[indices[0]], saved_X[index]) for index in indices[1:])
    np.testing.assert_allclose(rows["pki_target"].astype(float), y[indices], rtol=0, atol=1e-10)
    for record in rows.to_dict("records"):
        source_rows = retained.loc[retained["rdkit_smiles"].eq(record["rdkit_smiles"])]
        assert int(record["n_measurements"]) == len(source_rows)
        assert record["activity_ids"] == ";".join(sorted(source_rows["activity_id"]))
        assert record["assay_ids"] == ";".join(sorted(set(source_rows["assay_chembl_id"])))
        assert record["document_ids"] == ";".join(sorted(set(source_rows["document_chembl_id"]) - {""}))
        assert record["molecule_ids"] == ";".join(sorted(set(source_rows["molecule_chembl_id"])))
    # Recalculate the saved group sizes, pair counts, target spread, scaffold agreement, and gap flag.
    summary_row = saved_group_summary.set_index("fingerprint_group_id").loc[group_id]
    assert int(summary_row["structure_count"]) == len(indices)
    assert int(summary_row["unique_key_count"]) == len(set(rows["rdkit_smiles"]))
    assert int(summary_row["unordered_pair_count"]) == len(indices) * (len(indices) - 1) // 2
    for field, expected in (
        ("target_min", min(y[indices])), ("target_max", max(y[indices])),
        ("target_range", max(y[indices]) - min(y[indices])),
    ):
        assert np.isclose(float(summary_row[field]), expected, rtol=0, atol=1e-10), (group_id, field)
    assert (summary_row["same_scaffold"] == "True") == (len({scaffolds[index] for index in indices}) == 1)
    assert (summary_row["target_gap_ge_1"] == "True") == (
        max(y[indices]) - min(y[indices]) >= IDENTICAL_FP_TARGET_GAP
    )
print("Saved arrays, fingerprint bits, group membership, evidence links, and assignments verified.")

Saved arrays, fingerprint bits, group membership, evidence links, and assignments verified.


The boundary reports must still describe the frozen assignments after serialization. This cell checks exact fingerprint matches and recalculates each held-out structure's nearest-training similarity without using prediction performance. Only after these reports pass does the next cell record completion.

In [13]:
# Reload pair and neighbor reports with explicit text parsing before interpreting numeric fields.
saved_pairs = pd.read_csv(cross_pairs_path, dtype=str, keep_default_na=False)
saved_neighbors = pd.read_csv(neighbors_path, dtype=str, keep_default_na=False)
assert list(saved_pairs.columns) == pair_columns
assert list(saved_neighbors.columns) == neighbor_columns
# Rebuild the expected cross-boundary pair set and check both completeness and absence of duplicates.
expected_pairs = set()
for strategy in STRATEGIES:
    for left_subset, right_subset in combinations(SUBSETS, 2):
        for group_id, members in indices_by_group.items():
            left = [index for index in members if subset_for_key[strategy][structure_keys[index]] == left_subset]
            right = [index for index in members if subset_for_key[strategy][structure_keys[index]] == right_subset]
            for left_index in left:
                for right_index in right:
                    expected_pairs.add((strategy, left_subset, right_subset,
                                        structure_keys[left_index], structure_keys[right_index], group_id))
saved_pair_tuples = set(saved_pairs.itertuples(index=False, name=None))
assert len(saved_pairs) == len(saved_pair_tuples)
assert saved_pair_tuples == expected_pairs, "Saved cross-boundary fingerprint pairs are incomplete or incorrect."
# Verify each saved pair against actual feature bits and the frozen subset assignments.
for record in saved_pairs.to_dict("records"):
    left_index = key_to_index[record["left_rdkit_smiles"]]
    right_index = key_to_index[record["right_rdkit_smiles"]]
    assert np.array_equal(saved_X[left_index], saved_X[right_index])
    assert subset_for_key[record["split_strategy"]][record["left_rdkit_smiles"]] == record["left_subset"]
    assert subset_for_key[record["split_strategy"]][record["right_rdkit_smiles"]] == record["right_subset"]

# Require one record per held-out key, then recompute nearest similarities and deterministic tie choices.
assert not saved_neighbors.duplicated(["split_strategy", "subset", "rdkit_smiles"]).any()
assert len(saved_neighbors) == sum(
    len(split_indices[strategy][subset]) for strategy in STRATEGIES for subset in ("validation", "test")
)
for record in saved_neighbors.to_dict("records"):
    strategy, subset, key = record["split_strategy"], record["subset"], record["rdkit_smiles"]
    assert subset in ("validation", "test") and subset_for_key[strategy][key] == subset
    train_indices = split_indices[strategy]["train"]
    similarities = np.asarray(DataStructs.BulkTanimotoSimilarity(
        fingerprints[key_to_index[key]], [fingerprints[index] for index in train_indices]
    ))
    maximum = float(similarities.max())
    tied = np.flatnonzero(similarities == maximum)
    assert np.isclose(float(record["maximum_tanimoto"]), maximum, rtol=0, atol=1e-12)
    assert record["nearest_training_rdkit_smiles"] == structure_keys[train_indices[tied[0]]]
    assert int(record["nearest_tie_count"]) == len(tied)
    assert (record["exact_training_fingerprint_match"] == "True") == (maximum == 1.0)

# Check saved JSON equality and reconcile pair and neighbor counts against their detailed saved reports.
with audit_summary_path.open(encoding="utf-8") as stream:
    saved_audit = json.load(stream)
assert saved_audit == audit_summary
for boundary in saved_audit["audit_b"]["boundaries"]:
    matching = saved_pairs.loc[
        saved_pairs["split_strategy"].eq(boundary["split_strategy"])
        & saved_pairs["left_subset"].eq(boundary["left_subset"])
        & saved_pairs["right_subset"].eq(boundary["right_subset"])
    ]
    assert len(matching) == boundary["identical_fingerprint_pairs"]
    assert matching["left_rdkit_smiles"].nunique() == boundary["affected_left_structures"]
    assert matching["right_rdkit_smiles"].nunique() == boundary["affected_right_structures"]
for summary in saved_audit["audit_b"]["nearest_training_similarity"]:
    rows = saved_neighbors.loc[
        saved_neighbors["split_strategy"].eq(summary["split_strategy"])
        & saved_neighbors["subset"].eq(summary["subset"])
    ]
    values = rows["maximum_tanimoto"].astype(float).to_numpy()
    assert len(values) == summary["structures"]
    assert int(np.count_nonzero(values == 1.0)) == summary["exact_training_matches"]
    assert int(np.count_nonzero(values >= HIGH_SIMILARITY)) == summary["high_similarity_structures"]
print("Saved boundary pairs, nearest-training similarities, and audit counts verified.")

Saved boundary pairs, nearest-training similarities, and audit counts verified.


## 7. Record the completed preparation handoff

The completion manifest names the curation source, immutable preparation files, representation and split settings, audit findings, package versions, and Git state. It is written last, after every saved-file check and a final source rehash. Downstream model notebooks can use this manifest without rerunning curation or splitting.

In [14]:
# Recheck every upstream file so the source did not change during preparation.
for relative_name, recorded_hash in source_hashes.items():
    assert sha256_file(project_root / relative_name) == recorded_hash, (
        f"Upstream source changed during preparation: {relative_name}"
    )
assert sha256_file(curation_manifest_path) == curation_manifest_hash

# Trust this checkout only for these two read-only Git commands on a WSL UNC path.
git_command = ["git", "-c", f"safe.directory={project_root.as_posix()}"]
git_commit_result = subprocess.run(
    git_command + ["rev-parse", "HEAD"], cwd=project_root, capture_output=True, text=True, check=False
)
git_status_result = subprocess.run(
    git_command + ["status", "--porcelain"], cwd=project_root, capture_output=True, text=True, check=False
)
git_record = {
    "commit": git_commit_result.stdout.strip() if git_commit_result.returncode == 0 else None,
    "working_tree_dirty": bool(git_status_result.stdout.strip()) if git_status_result.returncode == 0 else None,
}
artifact_paths = [
    features_path, group_summary_path, group_members_path, assignment_path, split_manifest_path,
    cross_pairs_path, neighbors_path, audit_summary_path,
]
prep_manifest = {
    "run_started_at_utc": run_started_at_utc,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "notebooks/03_modeling_preparation.ipynb",
    "stage": "modeling_preparation",
    "output_folder": modeling_folder.relative_to(project_root).as_posix(),
    "source_curation_manifest": CURATION_MANIFEST_REL.as_posix(),
    "source_selection_manifest": curation_manifest["selection_manifest"],
    "source_acquisition_manifest": curation_manifest["acquisition_manifest"],
    "source_sha256": {**source_hashes, CURATION_MANIFEST_REL.as_posix(): curation_manifest_hash},
    "artifact_sha256": {
        path.relative_to(project_root).as_posix(): sha256_file(path) for path in artifact_paths
    },
    "target_policy": curation_manifest["policies"]["target_aggregation"],
    "structure_policy": curation_manifest["policies"]["structure_key"],
    "configuration": configuration,
    "counts": {
        "structures": len(structure_keys), "retained_measurements": len(retained),
        "scaffold_groups": len(set(scaffolds)),
        "identical_fingerprint_groups": len(group_summary),
        "identical_fingerprint_structures": len(group_members),
        "split_subsets": split_manifest["actual_proportions"],
    },
    "audit": saved_audit,
    "verification": {
        "source_hashes": True, "saved_arrays_and_fingerprint_bits": True,
        "saved_splits_and_scaffolds": True, "saved_audit_reports": True,
    },
    "models_fitted": False,
    "predictions_performed": False,
    "test_evaluation_performed": False,
    "software_versions": {
        "python": platform.python_version(), "platform": platform.system(), "rdkit": rdkit.__version__,
        "numpy": np.__version__, "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "git": git_record,
}
prep_manifest_path = modeling_folder / "manifest.json"
assert not prep_manifest_path.exists(), "Preparation completion manifest already exists."
with prep_manifest_path.open("w", encoding="utf-8") as stream:
    json.dump(prep_manifest, stream, indent=2)
assert prep_manifest_path.is_file()
print(f"Verified eight preparation artifacts and all recorded source hashes.")
print(f"Completed preparation run: {modeling_folder.relative_to(project_root).as_posix()}")

Verified eight preparation artifacts and all recorded source hashes.
Completed preparation run: provenance/preparation/run_01


## Handoff to model notebooks

The completed folder contains aligned binary fingerprints and targets, frozen random and scaffold assignments, and both audit reports. The random forest and dummy notebooks each fit their own estimators on these same training subsets. Test subsets remain reserved within each protocol.

## Refresh the project reference
After this stage passes its checks, update the model comparison and project file guide. The presentation workbook stays unchanged. Existing model results remain labeled with the dataset that produced them.


In [15]:
# Verify saved evidence and refresh Markdown references; preserve the presentation workbook.
from build_project_reference import build_reference
build_reference()


{
  "checksum_references": 217,
  "recomputed_metric_rows": 76,
  "active_notebooks": 10,
  "model_comparison_rows": 22,
  "current_pipeline": {
    "acquisition": "provenance/raw/chembl_cb2_20260922T165756862384Z/acquisition_manifest.json",
    "selection": "provenance/selection/chembl_cb2_20260922T165756862384Z/run_01/selection_manifest.json",
    "curation": "provenance/curation/run_01/manifest.json",
    "preparation": "provenance/preparation/run_01/manifest.json"
  },
  "reports_written": true,
  "presentation_workbook": "preserved unchanged"
}


{'checksum_references': 217,
 'recomputed_metric_rows': 76,
 'active_notebooks': 10,
 'model_comparison_rows': 22,
 'current_pipeline': {'acquisition': 'provenance/raw/chembl_cb2_20260922T165756862384Z/acquisition_manifest.json',
  'selection': 'provenance/selection/chembl_cb2_20260922T165756862384Z/run_01/selection_manifest.json',
  'curation': 'provenance/curation/run_01/manifest.json',
  'preparation': 'provenance/preparation/run_01/manifest.json'},
 'reports_written': True,
 'presentation_workbook': 'preserved unchanged'}